# RNA3D Pilot10 — Historical RNA structural-template **panel screening** (CPU only)

**Corrected v2:** Fixes a regex backspace-character defect; refuses to finalize incomplete scoring. The old v1 run is preserved.

**Stage:** controlled, bounded candidate discovery for structural fold analogues that were publicly released **on/before 2022-12-31 or 2023-12-31**. This is a **screening panel, not an exhaustive historical PDB search and not model-checkpoint contamination clearance**. An empty hit list only means *not found in the sampled panel*.

**Previous evidence:** 10 frozen RNAs; 8 native-chain provenance PASS and 2 REVIEW due to low coordinate coverage; all ten official PDB release dates after 2023; RCSB RNA **sequence** search at 30% identity reported zero pre-2024 hits. **Structural similarity can exist despite sequence divergence.**

**Key scoring distinction:** To compare unrelated RNA structures, US-align must **discover** structural correspondence; do **not** pass `-TMscore 1` (that option forces residue-number correspondence). Here we run `USalign candidate.pdb target_native.pdb -mol RNA`, choose the **Structure_2/native-normalized** TM-score, and report aligned coverage. For earlier *same-sequence prediction versus experimental target*, fixed-correspondence scoring used `-TMscore 1` and is a different condition.

**Sample design:** candidate RNA polymer entities selected from RCSB across three RNA length bins and four historical release-date strata (up to 5 per stratum, ≤60 nominal). This is a nonrandom, bounded **discovery panel**. All source query parameters, selection limits, failures and exclusions are recorded. Large complex CIF files are skipped with reasons. Do not claim absence of historical templates based on this screen.

No GPU, no model inference, no GitHub writes, no alteration of prior results.

## 1 — Setup and Drive mount

In [ ]:
!pip -q install 'biopython>=1.83,<2' requests pandas
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import csv, hashlib, json, os, re, subprocess, time, shutil, requests, math
import pandas as pd
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
EVID=ROOT/'Evaluations/pilot10_rcsb_release_historical_sequence_candidates_20261113_v1'
NATIVE=ROOT/'native/pilot_v1/native_eval'
AUDIT=ROOT/'Evaluations/pilot10_native_chain_provenance_20261113_v1'
RUN_ID='pilot10_historical_structural_panel_20261114_v2'
OUT=ROOT/'Evaluations'/RUN_ID
WORK=Path('/content/rna3d_historical_structural_panel')
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
# Conservative bounded discovery: each length × release-era stratum contributes ≤5 entities.
LENGTH_BINS=[(25,44),(45,69),(70,110)]
ERAS=[('through_2010',None,'2010-12-31'),('2011_2018','2011-01-01','2018-12-31'),('2019_2022','2019-01-01','2022-12-31'),('2023','2023-01-01','2023-12-31')]
MAX_PER_STRATUM=5
MAX_CIF_BYTES=12_000_000
MAX_TARGETS_PER_CANDIDATE=10
SEARCH_URL='https://search.rcsb.org/rcsbsearch/v2/query'
DATA_URL='https://data.rcsb.org/rest/v1/core'
CACHE=WORK/'raw_cif'; STRUCT=WORK/'candidate_pdb'; CACHE.mkdir(parents=True,exist_ok=True); STRUCT.mkdir(parents=True,exist_ok=True)
assert EVID.is_dir() and NATIVE.is_dir() and AUDIT.is_dir(), 'Missing frozen Drive sources'
assert not OUT.exists(),f'Output {OUT} already exists. Never overwrite a completed historical panel.'
assert len(TARGETS)==len(set(TARGETS))==10
print('CPU-only panel. Candidate limit:',len(LENGTH_BINS)*len(ERAS)*MAX_PER_STRATUM)
print('No existing outputs will be overwritten. Destination:',OUT)


## 2 — Verify pinned source evidence and native PDB references

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def load_rows(path):
    with open(path,newline='') as f:return list(csv.DictReader(f))
def check_final(folder):
    obj=json.loads((folder/'FINALIZED.json').read_text());assert obj['status']=='complete'
    for name,digest in obj['files'].items():
        assert sha256(folder/name)==digest,(folder,name)
    return obj
src=check_final(EVID)
audit_final=check_final(AUDIT)
dates=load_rows(EVID/'official_release_dates.csv')
assert {x['target_id'] for x in dates}==set(TARGETS)
audit={x['target_id']:x for x in load_rows(AUDIT/'native_chain_audit.csv')}
assert set(audit)==set(TARGETS)
for t in TARGETS:
    assert (NATIVE/f'{t}_native_eval.pdb').is_file()
    assert audit[t]['status'] in ('PASS','REVIEW')
    assert dates[[q['target_id'] for q in dates].index(t)]['official_initial_release_date']>'2023-12-31'
print('SOURCE EVIDENCE VERIFIED; native audit:',{s:sum(audit[t]['status']==s for t in TARGETS) for s in ['PASS','REVIEW']})


## 3 — Pin and build US-align; **free structural alignment** without `-TMscore 1`

In [ ]:
USDIR=Path('/content/USalign-rna3d-historical-panel')
if not USDIR.exists():subprocess.run(['git','clone','--depth','1','--branch','20260527','https://github.com/pylelab/USalign.git',str(USDIR)],check=True)
commit=subprocess.check_output(['git','-C',str(USDIR),'rev-parse','HEAD'],text=True).strip()
tagcommit=subprocess.check_output(['git','-C',str(USDIR),'rev-parse','refs/tags/20260527^{commit}'],text=True).strip()
assert commit==tagcommit
US=USDIR/'USalign'
if not US.exists():subprocess.run(['make','-j2'],cwd=USDIR,check=True)
assert US.is_file()
print('US-align version 20260527, commit',commit,'SHA256',sha256(US))


## 4 — RCSB RNA-entity discovery with explicit historical and length strata

In [ ]:
session=requests.Session();session.headers.update({'User-Agent':'RNA3D-template-scarcity-academic-pilot/1.0','Accept':'application/json'})
def get_json(url,params=None):
    for retry in range(4):
        try:
            r=session.get(url,params=params,timeout=45)
            if r.status_code==204:return {'total_count':0,'result_set':[]}
            r.raise_for_status();return r.json()
        except (requests.RequestException,ValueError):
            if retry==3:raise
            time.sleep(2**retry)

def term(attr,op,value):
    return {'type':'terminal','service':'text','parameters':{'attribute':attr,'operator':op,'value':value}}
def strata_query(lo,hi,start,end):
    nodes=[term('entity_poly.rcsb_entity_polymer_type','exact_match','RNA'),
           term('entity_poly.rcsb_sample_sequence_length','greater_or_equal',lo),
           term('entity_poly.rcsb_sample_sequence_length','less_or_equal',hi),
           term('rcsb_accession_info.initial_release_date','less_or_equal',end),
           term('rcsb_entry_info.structure_determination_methodology','exact_match','experimental')]
    if start:nodes.append(term('rcsb_accession_info.initial_release_date','greater_or_equal',start))
    return {'query':{'type':'group','logical_operator':'and','nodes':nodes},
            'return_type':'polymer_entity',
            'request_options':{'paginate':{'start':0,'rows':MAX_PER_STRATUM},
              'sort':[{'sort_by':'rcsb_accession_info.initial_release_date','direction':'desc'}]}}

picked=[];strata=[]
for lo,hi in LENGTH_BINS:
  for era,start,end in ERAS:
    key=f'{era}_{lo}_{hi}'
    payload=strata_query(lo,hi,start,end)
    # GET + compact JSON request; no API key required.
    response=get_json(SEARCH_URL,{'json':json.dumps(payload,separators=(',',':'))})
    total=int(response.get('total_count') or 0)
    hit=[r['identifier'] for r in response.get('result_set',[])]
    if not isinstance(hit,list):raise RuntimeError('Unexpected RCSB search format')
    strata.append({'stratum':key,'low_length':lo,'high_length':hi,'release_start':start or '',
       'release_end':end,'reported_total':total,'returned':len(hit),
       'truncated_to_limit':total>len(hit),'search_status':'COMPLETE'})
    for entity in hit:picked.append({'entity_id':entity,'stratum':key,'release_cutoff':end})
    print(key,'total',total,'sampled',len(hit))
    time.sleep(0.12)
assert len(strata)==len(LENGTH_BINS)*len(ERAS)
# De-duplicate across possible inter-query overlap, preserve discovery order.
selection=list({x['entity_id']:x for x in reversed(picked)}.values())[::-1]
assert len(selection)<=len(strata)*MAX_PER_STRATUM
print('Unique selected polymer entities:',len(selection),'of all possible RCSB RNA entities; bounded, NOT exhaustive')


## 5 — Collect entity metadata and safely extract one RNA chain per entity from raw mmCIF

In [ ]:
def parse_pdb_entity_id(entity):
    m=re.fullmatch(r'([0-9A-Z]{4})_([0-9]+)',entity)
    if not m:raise ValueError('Unrecognized entity identifier: '+entity)
    return m.group(1),m.group(2)

def cif_download(pdb):
    path=CACHE/f'{pdb}.cif'
    if path.exists():return path
    url=f'https://files.rcsb.org/download/{pdb}.cif'
    for attempt in range(3):
      try:
        r=session.get(url,stream=True,timeout=90);r.raise_for_status()
        declared=int(r.headers.get('Content-Length','0'))
        if declared>MAX_CIF_BYTES:raise ValueError(f'CIF_TOO_LARGE_DECLARED:{declared}')
        tmp=path.with_suffix('.part');size=0
        with tmp.open('wb') as f:
          for chunk in r.iter_content(chunk_size=1<<18):
            if not chunk:continue
            size+=len(chunk)
            if size>MAX_CIF_BYTES:raise ValueError(f'CIF_TOO_LARGE_STREAMED:{size}')
            f.write(chunk)
        tmp.replace(path);return path
      except Exception as e:
        path.with_suffix('.part').unlink(missing_ok=True)
        if isinstance(e,ValueError):raise
        if attempt==2:raise
        time.sleep(attempt+1)

def vals(dic,key):return dic.get(key,[])
BASES={'A':'A','C':'C','G':'G','U':'U','RA':'A','RC':'C','RG':'G','RU':'U','ADE':'A','CYT':'C','GUA':'G','URA':'U'}

def extract_entity_c3(cif_path,entityid,output):
    d=MMCIF2Dict(str(cif_path))
    labels=vals(d,'_atom_site.label_entity_id')
    asym=vals(d,'_atom_site.label_asym_id')
    atomnames=vals(d,'_atom_site.label_atom_id')
    seqids=vals(d,'_atom_site.label_seq_id')
    names=vals(d,'_atom_site.label_comp_id')
    xyz=[vals(d,'_atom_site.Cartn_x'),vals(d,'_atom_site.Cartn_y'),vals(d,'_atom_site.Cartn_z')]
    models=vals(d,'_atom_site.pdbx_PDB_model_num')
    alts=vals(d,'_atom_site.label_alt_id')
    assert len(labels)==len(asym)==len(atomnames)==len(seqids)==len(names)==len(xyz[0])==len(xyz[1])==len(xyz[2])
    chains=sorted({asym[i] for i,e in enumerate(labels) if e==entityid and seqids[i] not in ('.','?')})
    if not chains:raise ValueError('ENTITY_HAS_NO_COORDINATE_CHAIN')
    chain=chains[0]; observed={}
    for i,e in enumerate(labels):
      if e!=entityid or asym[i]!=chain or atomnames[i].replace('*',"'")!="C3'":continue
      if models and models[i]!='1':continue
      if alts and alts[i] not in ('.','?','A','1'):continue
      if not seqids[i].isdigit():continue
      base=BASES.get(names[i].upper())
      if base is None:continue
      pos=int(seqids[i]);
      if pos not in observed:observed[pos]=(base,*[float(c[i]) for c in xyz])
    if len(observed)<10:raise ValueError('TOO_FEW_C3_COORDINATES:'+str(len(observed)))
    lines=[]
    for serial,(pos,(base,x,y,z)) in enumerate(sorted(observed.items()),1):
      lines.append(f"ATOM  {serial:5d}  C3' {base:3s} A{pos:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.0:6.2f}{0.0:6.2f}           C")
    output.write_text('\n'.join(lines+['TER','END'])+'\n')
    return {'label_asym_id':chain,'observed_C3_residues':len(lines),
            'first_label_seq_id':min(observed),'last_label_seq_id':max(observed)}

metadata=[]; usable=[]
for idx,item in enumerate(selection,1):
    entity=item['entity_id'];pdb,ent=parse_pdb_entity_id(entity)
    rec={'entity_id':entity,'pdb_id':pdb,'stratum':item['stratum'],'status':'','reason':'','source_url':f'https://files.rcsb.org/download/{pdb}.cif'}
    try:
      entry=get_json(f'{DATA_URL}/entry/{pdb}')
      release=entry['rcsb_accession_info']['initial_release_date'][:10]
      rec['initial_release_date']=release
      assert release<='2023-12-31',f'NONHISTORICAL_RELEASE:{release}'
      emeta=get_json(f'{DATA_URL}/polymer_entity/{pdb}/{ent}')
      rec['sample_length']=int(emeta['entity_poly']['rcsb_sample_sequence_length'])
      assert 25<=rec['sample_length']<=110
      path=cif_download(pdb)
      outfile=STRUCT/f'{entity}.pdb'
      info=extract_entity_c3(path,ent,outfile)
      rec.update(info);rec['status']='EXTRACTED';rec['raw_cif_sha256']=sha256(path)
      rec['candidate_pdb_sha256']=sha256(outfile)
      usable.append(rec)
    except Exception as exc:
      rec['status']='SKIPPED';rec['reason']=str(exc)[:250]
    metadata.append(rec)
    if idx%10==0 or idx==len(selection): print(f'{idx}/{len(selection)} processed; usable={len(usable)}')
assert len(metadata)==len(selection)
if not usable:raise RuntimeError('No extractable historical RNA candidates: inspect metadata failures before scoring.')
print('Historical RNA candidates available for screening:',len(usable),'\nNOTE: panel is non-exhaustive.')


## 6 — Structural alignment (not fixed correspondence); score candidate vs all 10 native targets

In [ ]:
def score_usalign(candidate,native):
    # Different molecules/sequences require free structural alignment. No -TMscore 1.
    cmd=[str(US),str(candidate),str(native),'-mol','RNA']
    p=subprocess.run(cmd,capture_output=True,text=True,timeout=120)
    if p.returncode:raise RuntimeError(f'USalign exit={p.returncode}: {p.stderr[-200:]}')
    s=p.stdout
    def find(pattern):
        m=re.search(pattern,s,re.I)
        if not m:raise ValueError('Missing US-align field: '+pattern)
        return m
    aligned=find(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([0-9.eE+-]+)')
    tm=find(r'TM-score\s*=\s*([0-9.eE+-]+)\s*\(normalized by length of Structure_2\b')
    leng=find(r'Length of Structure_2:\s*(\d+) residues')
    n=int(leng.group(1));k=int(aligned.group(1));v=float(tm.group(1))
    if not(0<k<=n and 0<=v<=1 and math.isfinite(float(aligned.group(2)))):raise ValueError('Invalid alignment metrics')
    return {'tm_native_normalized':v,'rmsd_angstrom':float(aligned.group(2)),
            'aligned_residues':k,'native_residues':n,'native_coverage':k/n}

# Exact regression test for the native-normalized score label.
_test_output = """Length of Structure_2: 51 residues
Aligned length= 32, RMSD= 5.70, Seq_ID=n_identical/n_aligned= 0.231
TM-score= 0.29123 (normalized by length of Structure_1: L=36, d0=0.88)
TM-score= 0.25345 (normalized by length of Structure_2: L=51, d0=1.76)"""
_test_match = re.search(r'TM-score\s*=\s*([0-9.eE+-]+)\s*\(normalized by length of Structure_2\b', _test_output)
assert _test_match and _test_match.group(1)=='0.25345', 'US-align parser regression failed'
print('US-align Structure_2 parser regression test: PASS')

hits=[];failures=[]
for i,u in enumerate(usable,1):
  pred=STRUCT/f"{u['entity_id']}.pdb"
  for target in TARGETS:
    native=NATIVE/f'{target}_native_eval.pdb'
    try:
      sc=score_usalign(pred,native)
      hits.append({'target_id':target,'candidate_entity_id':u['entity_id'],
                   'candidate_pdb_id':u['pdb_id'],'candidate_release_date':u['initial_release_date'],
                   'candidate_observed_residues':u['observed_C3_residues'],
                   'native_audit_status':audit[target]['status'],
                   'alignment_mode':'FREE_STRUCTURE_ALIGNMENT_NOT_FIXED_NUMBERING',**sc})
    except Exception as e:failures.append({'target_id':target,'entity_id':u['entity_id'],'error':str(e)[:250]})
  if i%10==0 or i==len(usable):print(f'{i}/{len(usable)} candidates compared; scored {len(hits)}; errors {len(failures)}')
assert len(hits)+len(failures)==len(usable)*len(TARGETS)
if not hits or failures:
    raise RuntimeError(f'SCORING INCOMPLETE: {len(hits)} successes / {len(failures)} failures. '
                       f'Examples: {failures[:4]}')
print('HISTORICAL SCORING VERIFIED:',len(hits),'/',len(usable)*len(TARGETS))


## 7 — Analysis: aligned coverage, score thresholds, and honest scope statement

In [ ]:
frame=pd.DataFrame(hits)
if len(frame):
    frame['coverage_ge_0p50']=frame['native_coverage']>=0.5
    frame['tm_ge_0p45']=frame['tm_native_normalized']>=0.45
    frame['candidate_flag_for_review']=frame['coverage_ge_0p50'] & frame['tm_ge_0p45']
    display(frame.sort_values(['target_id','tm_native_normalized'],ascending=[True,False]).groupby('target_id').head(3))
summary_rows=[]
for target in TARGETS:
    cur=frame.loc[frame.target_id==target] if len(frame) else pd.DataFrame()
    eligible=cur[cur.native_coverage>=0.5] if len(cur) else cur
    maximum=float(eligible.tm_native_normalized.max()) if len(eligible) else None
    summary_rows.append({'target_id':target,'native_audit_status':audit[target]['status'],
      'panel_entities_scored':len(cur),'coverage_ge_0p50_candidates':len(eligible),
      'highest_tm_coverage_ge_0p50':maximum,'flagged_high_similarities':int((eligible.tm_native_normalized>=0.45).sum()) if len(eligible) else 0,
      'historical_template_scarcity_status':'UNDETERMINED_NONEXHAUSTIVE_PANEL'})
summary_df=pd.DataFrame(summary_rows);display(summary_df)
print('IMPORTANT: These panel results do not establish that a training template was or was not available.')
print('Even a high-score historical candidate is only a structural analogue, not proven model training membership.')


## 8 — Save a new immutable Drive snapshot with checksums and failure accounting

In [ ]:
assert len(hits)==len(usable)*len(TARGETS) and not failures and len(hits)>0, 'Refusing to finalize incomplete scoring'
assert not OUT.exists(),'Never overwrite a previously saved run.'
STAGE=WORK/'export'
if STAGE.exists():raise FileExistsError(f'Staging exists: {STAGE}; inspect before re-running')
STAGE.mkdir(parents=True)
pd.DataFrame(strata).to_csv(STAGE/'rcsb_query_strata.csv',index=False)
pd.DataFrame(metadata).to_csv(STAGE/'candidate_extraction_audit.csv',index=False)
pd.DataFrame(hits).to_csv(STAGE/'structural_panel_scores.csv',index=False)
pd.DataFrame(failures,columns=['target_id','entity_id','error']).to_csv(STAGE/'score_failures.csv',index=False)
summary_df.to_csv(STAGE/'per_target_panel_summary.csv',index=False)
meta={'status':'complete_with_limits','run_id':RUN_ID,'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_release_run':'pilot10_rcsb_release_historical_sequence_candidates_20261113_v1',
 'source_release_finalized_sha256':sha256(EVID/'FINALIZED.json'),
 'source_native_finalized_sha256':sha256(AUDIT/'FINALIZED.json'),
 'usalign_release':'20260527','usalign_commit':commit,'usalign_binary_sha256':sha256(US),
 'command':'USalign historical_candidate.pdb experimental_native.pdb -mol RNA',
 'no_fixed_correspondence':True,
 'candidate_selection':'Non-random bounded RNA entity length/date stratified panel, maximum five entities per stratum; non-exhaustive',
 'length_bins':LENGTH_BINS,'release_strata':ERAS,'max_per_stratum':MAX_PER_STRATUM,
 'max_download_bytes_per_cif':MAX_CIF_BYTES,
 'candidates_selected':len(selection),'candidates_extracted':len(usable),'candidates_skipped':len(metadata)-len(usable),
 'target_comparisons_scored':len(hits),'target_comparisons_failed':len(failures),
 'scoring_threshold_for_review':{'tm_native_normalized_min':0.45,'aligned_native_coverage_min':0.5},
 'unresolved':['Not a comprehensive PDB historical screen; nonrandom sample','Training structures of exact model checkpoints unverified',
               'Some RNA native targets have missing experimental coordinates','Candidate chain extraction uses one representative instance per polymer entity',
               'BGSU 2022 snapshot and historical structural nonredundant database memberships not validated',
               'Structural similarity alone cannot prove training leakage','No claims of template scarcity or leak-free evaluation']}
(STAGE/'summary.json').write_text(json.dumps(meta,indent=2)+'\n')
OUT.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(STAGE,OUT)
required=['rcsb_query_strata.csv','candidate_extraction_audit.csv','structural_panel_scores.csv','score_failures.csv','per_target_panel_summary.csv','summary.json']
for f in required:assert sha256(OUT/f)==sha256(STAGE/f)
(OUT/'FINALIZED.json').write_text(json.dumps({'status':'complete','run_id':RUN_ID,
  'files':{f:sha256(OUT/f) for f in required}},indent=2)+'\n')
print('HISTORICAL STRUCTURAL PANEL COMPLETE: candidates',len(usable),'comparisons',len(hits),'failures',len(failures))
print('SAVED, VERIFIED:',OUT)
print('Not exhaustive. No GPU. No inference. No GitHub write.')


## Interpretation rules for the report

- **No historical structural hits in this panel ≠ template scarcity proven.** This panel is deliberately bounded by date, RNA length, selected entities, download size and chain representation. `reported_total` and per-file skip reasons must accompany any interpretation.
- **High TM-score with reasonable aligned coverage** means a *possible structural analogue* was historically available; inspect whether it is genuine RNA fold similarity, complex-related biology or a metric artifact. It **does not** demonstrate training overlap.
- For small reference structures, native-normalized TM-score remains sensitive to reference length; always report native residue count, aligned coverage and RMSD.
- Next scientific stage: select a **complete, dated nonredundant historical RNA archive**, independently rerun the search with defined length/coverage/chain policies, and separately obtain model checkpoint-specific training/validation lists.
- `8ubd_H` and `9lq7_K` remain low-native-coverage REVIEW targets.

- The first **v1 run must not be cited** as valid structural scoring: a broken parser produced zero scored alignments. This v2 run requires complete scoring before saving.
